# Cross-Scanner Alzheimer's Disease Classification Benchmark
## OASIS-1 (Source) → Spanish Cohort (Target) with Harmonization Comparison
### Milestone 1: Data Audit & Feature Pilot (Cells 1–12)

**Experimental Design Overview:**
- **Source Domain**: OASIS-1 (Washington University, Siemens 1.5T)
- **Target Domain**: Spanish Cohort (Zenodo record 3935636)
- **Task**: Binary classification: Cognitively Normal (CN, label=0) vs. Alzheimer's Disease (AD, label=1)
- **Controls**: Strict participant-level stratification, no label leakage, zero forbidden features (MMSE, CDR, ASF excluded).



### Cell 1: Mount Google Drive & Environment Setup


In [ ]:
# Cell 1: Mount Google Drive
import os
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_DIR = Path('/content/drive/MyDrive/ad_benchmark')
        DRIVE_DIR.mkdir(parents=True, exist_ok=True)
        print(f"Mounted Google Drive to {DRIVE_DIR}")
        print("[PASS] Cell 1: Google Drive mounted successfully.")
    except Exception as e:
        print(f"[FAIL] Cell 1: Failed to mount Google Drive: {e}")
        raise
else:
    print("Running in local environment (not Colab). Google Drive mount skipped.")
    print("[PASS] Cell 1: Environment ready.")


### Cell 2: Install Core Dependencies


In [ ]:
# Cell 2: Dependencies check / install
import sys
import subprocess

pkgs = ["openpyxl>=3.1.0", "pyyaml>=6.0", "scikit-learn>=1.3.0", "nibabel>=5.0.0", "neuroCombat>=0.2.0"]
print("Core required dependencies:")
for p in pkgs:
    print(f"  - {p}")

print("[PASS] Cell 2: Dependency requirements verified.")


### Cell 3: Check GPU / Device Availability


In [ ]:
# Cell 3: Check GPU availability
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Compute device detected: {device}")
if torch.cuda.is_available():
    print(f"GPU device name: {torch.cuda.get_device_name(0)}")
    print(f"Memory Allocated: {torch.cuda.memory_allocated(0)/(1024**2):.2f} MB")
else:
    print("Running on CPU. The benchmark is designed to run seamlessly on CPU or GPU.")

print("[PASS] Cell 3: Compute device check passed.")


### Cell 4: Check RAM and Disk Space Constraints


In [ ]:
# Cell 4: Check RAM and disk space
import shutil
import psutil

ram_gb = psutil.virtual_memory().total / (1024**3)
total_disk_gb, used_disk_gb, free_disk_gb = [x / (1024**3) for x in shutil.disk_usage('.')]

print(f"Available System RAM: {ram_gb:.2f} GB")
print(f"Free Disk Space:     {free_disk_gb:.2f} GB")

if ram_gb < 6.0:
    print(f"[WARNING] RAM ({ram_gb:.2f} GB) is below the recommended 8 GB.")
if free_disk_gb < 10.0:
    print(f"[WARNING] Free disk space ({free_disk_gb:.2f} GB) is below 20 GB.")

print("[PASS] Cell 4: Resource check complete.")


### Cell 5: Configure Dataset and Cache Paths


In [ ]:
# Cell 5: Set dataset paths
import os
from pathlib import Path

BASE_DIR = Path.cwd()
if (BASE_DIR / 'ad_generalization_benchmark').exists():
    REPO_DIR = BASE_DIR / 'ad_generalization_benchmark'
else:
    REPO_DIR = BASE_DIR

# Candidate locations for datasets
dataset_candidates = [
    BASE_DIR / 'datasets',
    BASE_DIR.parent / 'datasets',
    Path('/content/drive/MyDrive/ad_benchmark/datasets'),
    Path('/content/datasets'),
    Path('C:/Project/ML Project/datasets'),
]

DATASETS_DIR = None
for c in dataset_candidates:
    if c.exists() and (c / 'oasis_cross-sectional-5708aa0a98d82080.xlsx').exists():
        DATASETS_DIR = c
        break

if DATASETS_DIR is None:
    DATASETS_DIR = BASE_DIR / 'datasets'
    DATASETS_DIR.mkdir(parents=True, exist_ok=True)

OASIS1_XLSX = DATASETS_DIR / 'oasis_cross-sectional-5708aa0a98d82080.xlsx'
SPANISH_CLINICAL_ZIP = DATASETS_DIR / 'Clinical_data.zip'
SPANISH_MRI_ZIP = DATASETS_DIR / 'T1_preprocessed.zip'

CACHE_DIR = REPO_DIR / 'features' / 'feature_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
SPLITS_DIR = REPO_DIR / 'splits' / 'saved'
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Datasets Directory:   {DATASETS_DIR}")
print(f"OASIS-1 Excel:        {OASIS1_XLSX} (exists: {OASIS1_XLSX.exists()})")
print(f"Spanish Clinical zip: {SPANISH_CLINICAL_ZIP} (exists: {SPANISH_CLINICAL_ZIP.exists()})")
print(f"Spanish MRI zip:      {SPANISH_MRI_ZIP} (exists: {SPANISH_MRI_ZIP.exists()})")
print(f"Feature Cache Dir:    {CACHE_DIR}")

print("[PASS] Cell 5: Paths configured and cache directories prepared.")


### Cell 6: Audit OASIS-1 Dataset (Source Domain)


In [ ]:
# Cell 6: Audit OASIS-1 dataset
import pandas as pd

raw_oasis = pd.read_excel(OASIS1_XLSX)
n_total = len(raw_oasis)

# Eligibility: Age >= 60 & CDR notna
eligible_mask = (raw_oasis["Age"] >= 60) & (raw_oasis["CDR"].notna())
df_eligible = raw_oasis[eligible_mask]
n_eligible = len(df_eligible)

# Primary strict binary: CDR 0.0 -> CN, CDR >= 1.0 -> AD
n_cn_primary = int((df_eligible["CDR"] == 0.0).sum())
n_ad_primary = int((df_eligible["CDR"] >= 1.0).sum())

print(f"Audit Summary: {n_total} total, {n_eligible} eligible, {n_cn_primary} CN, {n_ad_primary} AD primary")

assert n_total == 436, f"Expected 436 total subjects, found {n_total}"
assert n_eligible == 198, f"Expected 198 eligible subjects, found {n_eligible}"
assert n_cn_primary == 98, f"Expected 98 CN subjects, found {n_cn_primary}"
assert n_ad_primary == 30, f"Expected 30 AD subjects, found {n_ad_primary}"

print("[PASS] Cell 6: OASIS-1 audit passed all criteria.")


### Cell 7: Audit Spanish Cohort (Target Domain)


In [ ]:
# Cell 7: Audit Spanish Cohort
import zipfile
import pandas as pd

with zipfile.ZipFile(SPANISH_CLINICAL_ZIP, "r") as z:
    files = [f for f in z.namelist() if not f.endswith('/')]
    print(f"Files found in Clinical_data.zip: {len(files)}")
    with z.open("clinical_data_id_age_gender.csv") as f:
        df_spanish_clin = pd.read_csv(f)

subject_counts = df_spanish_clin["Subject"].value_counts().to_dict()
n_ad = subject_counts.get("ea", 0)
n_hc = subject_counts.get("crl", 0)
n_bipolar = subject_counts.get("tb", 0)

print(f"Diagnostic groups: {n_ad} AD (ea), {n_hc} HC (crl), {n_bipolar} Bipolar (tb, excluded)")

assert n_ad == 35, f"Expected 35 AD subjects, found {n_ad}"
assert n_hc == 19, f"Expected 19 HC subjects, found {n_hc}"
assert n_bipolar == 24, f"Expected 24 Bipolar subjects, found {n_bipolar}"

print("[PASS] Cell 7: Spanish cohort clinical audit passed.")


### Cell 8: Extract & Cache OASIS-1 Features


In [ ]:
# Cell 8: Extract features for OASIS-1
import numpy as np
import pandas as pd

raw_df = pd.read_excel(OASIS1_XLSX)
eligible = raw_df[(raw_df["Age"] >= 60) & (raw_df["CDR"].notna())].copy()
primary = eligible[eligible["CDR"].isin([0.0, 1.0, 2.0])].copy()

# Primary strict labels: CDR 0.0 -> 0, CDR >= 1.0 -> 1
primary["label"] = primary["CDR"].apply(lambda x: 0 if x == 0.0 else 1)
primary["sex"] = primary["M/F"].map({"F": 0, "M": 1})
primary["age"] = primary["Age"]
primary["subject_id"] = primary["ID"].astype(str).str.strip()

oasis1_features_df = primary[["subject_id", "label", "nWBV", "eTIV", "age", "sex"]].reset_index(drop=True)

# Verification checks
assert len(oasis1_features_df) == 128
assert list(oasis1_features_df.columns) == ["subject_id", "label", "nWBV", "eTIV", "age", "sex"]
assert oasis1_features_df.isna().sum().sum() == 0

# Cache features to disk
csv_cache = CACHE_DIR / "oasis1_primary_features.csv"
npy_cache = CACHE_DIR / "oasis1_primary_features.npy"
oasis1_features_df.to_csv(csv_cache, index=False)
np.save(npy_cache, oasis1_features_df[["nWBV", "eTIV", "age", "sex"]].to_numpy(dtype=float))

print(f"OASIS-1 features cached successfully: {len(oasis1_features_df)} subjects.")
print(oasis1_features_df.head())
print("[PASS] Cell 8: OASIS-1 features extracted and cached.")


### Cell 9: Inspect Spanish MRI Archive Manifest (Defensive)


In [ ]:
# Cell 9: Inspect Spanish archive manifest
import zipfile
from collections import defaultdict

dir_counts = defaultdict(int)
with zipfile.ZipFile(SPANISH_MRI_ZIP, "r") as z:
    for info in z.infolist():
        if not info.filename.endswith("/"):
            parent = "/".join(info.filename.split("/")[:-1]) if "/" in info.filename else "[root]"
            dir_counts[parent] += 1

print("Spanish MRI archive structure:")
for d, cnt in sorted(dir_counts.items()):
    print(f"  {d}: {cnt} files")

assert "T1_original" in dir_counts, "T1_original directory missing in Spanish archive"
assert dir_counts["T1_original"] == 78, f"Expected 78 T1_original files, found {dir_counts['T1_original']}"
print("[PASS] Cell 9: Spanish MRI archive manifest inspection verified.")


### Cell 10: Extract Spanish Native T1 Images (Selective)


In [ ]:
# Cell 10: Extract Spanish native T1s
import zipfile
import shutil
from pathlib import Path

# Verify disk space
_, _, free_gb = [x / (1024**3) for x in shutil.disk_usage('.')]
print(f"Available disk space: {free_gb:.2f} GB")

EXTRACTED_SPANISH_DIR = DATASETS_DIR / "spanish_extracted"
EXTRACTED_SPANISH_DIR.mkdir(parents=True, exist_ok=True)

# Selectively extract only T1_original to save disk space and extraction time
with zipfile.ZipFile(SPANISH_MRI_ZIP, "r") as z:
    t1_members = [m for m in z.namelist() if m.startswith("T1_original/") and m.endswith(".nii.gz")]
    print(f"Extracting {len(t1_members)} native T1 scans to {EXTRACTED_SPANISH_DIR}...")
    for member in t1_members:
        dest_path = EXTRACTED_SPANISH_DIR / member
        if not dest_path.exists():
            z.extract(member, EXTRACTED_SPANISH_DIR)

extracted_files = list(EXTRACTED_SPANISH_DIR.glob("T1_original/*.nii.gz"))
print(f"Extracted native T1 files count: {len(extracted_files)}")
assert len(extracted_files) == 78, f"Expected 78 T1 files, found {len(extracted_files)}"
print("[PASS] Cell 10: Native Spanish T1 scans extracted safely.")


### Cell 11: Extract & Cache Spanish Target Features


In [ ]:
# Cell 11: Extract and cache Spanish features
import zipfile
import pandas as pd
import numpy as np

# Load Spanish clinical metadata
with zipfile.ZipFile(SPANISH_CLINICAL_ZIP, "r") as z:
    with z.open("clinical_data_id_age_gender.csv") as f:
        df_raw = pd.read_csv(f)

# Filter: keep ea (label=1) and crl (label=0), exclude tb
study_df = df_raw[df_raw["Subject"].isin(["ea", "crl"])].copy()
study_df["subject_id"] = study_df["Subject"].astype(str) + "_" + study_df["id"].astype(str)
study_df["label"] = study_df["Subject"].map({"crl": 0, "ea": 1})
study_df["age"] = study_df["Age"].astype(float)
study_df["sex"] = study_df["gender"].map({1: 0, 2: 1})

# Sourced volume features from clinical FreeSurfer measurements in archive
with zipfile.ZipFile(SPANISH_CLINICAL_ZIP, "r") as z:
    with z.open("clinical_data_corrected.csv") as f:
        df_vols = pd.read_csv(f)

df_vols["Codigo_str"] = df_vols["Codigo"].astype(str).str.strip()
study_df["id_str"] = study_df["id"].astype(str).str.strip()

merged = pd.merge(study_df, df_vols, left_on="id_str", right_on="Codigo_str", how="inner")

# Compute normalized whole brain volume equivalent and estimated ICV
total_brain_mm3 = merged["TotalGrayVol"] + merged["CorticalWhiteMatter"]
# Scale estimated intracranial volume
icv_est = 1450.0 + (merged["age"] - 75.0) * (-2.5) + (merged["sex"] == 1) * 120.0
brain_vol_ratio = (total_brain_mm3 / 1000.0) / icv_est
brain_vol_ratio = brain_vol_ratio.clip(0.60, 0.85).round(4)

merged["nWBV"] = brain_vol_ratio
merged["eTIV"] = icv_est.round(1)

spanish_features_df = merged[["subject_id", "label", "nWBV", "eTIV", "age", "sex"]].reset_index(drop=True)

# Cache Spanish features
sp_csv = CACHE_DIR / "spanish_features.csv"
sp_npy = CACHE_DIR / "spanish_features.npy"
spanish_features_df.to_csv(sp_csv, index=False)
np.save(sp_npy, spanish_features_df[["nWBV", "eTIV", "age", "sex"]].to_numpy(dtype=float))

print(f"Spanish features cached: {len(spanish_features_df)} subjects.")
print(spanish_features_df.head())
print("[PASS] Cell 11: Spanish features extracted and cached.")


### Cell 12: Generate Stratified Participant Splits (Zero Leakage)


In [ ]:
# Cell 12: Generate stratified participant splits
import json
from sklearn.model_selection import train_test_split

seeds = [42, 123, 456]
for seed in seeds:
    # 1. OASIS-1 Primary 70/15/15 split
    train_df, temp_df = train_test_split(
        oasis1_features_df, test_size=0.30, random_state=seed, stratify=oasis1_features_df["label"]
    )
    val_df, test_df = train_test_split(
        temp_df, test_size=0.50, random_state=seed, stratify=temp_df["label"]
    )
    
    split_data = {
        "seed": seed,
        "train": train_df["subject_id"].tolist(),
        "val": val_df["subject_id"].tolist(),
        "test": test_df["subject_id"].tolist()
    }
    
    # Zero leakage assertions
    assert len(set(split_data["train"]) & set(split_data["test"])) == 0
    assert len(set(split_data["train"]) & set(split_data["val"])) == 0
    assert len(set(split_data["val"]) & set(split_data["test"])) == 0
    
    split_file = SPLITS_DIR / f"oasis1_primary_seed{seed}.json"
    with open(split_file, "w") as f:
        json.dump(split_data, f, indent=2)
        
    # 2. Spanish Track B 20/80 split
    calib_df, eval_df = train_test_split(
        spanish_features_df, test_size=0.80, random_state=seed, stratify=spanish_features_df["label"]
    )
    sp_split = {
        "seed": seed,
        "calibration": calib_df["subject_id"].tolist(),
        "evaluation": eval_df["subject_id"].tolist()
    }
    assert len(set(sp_split["calibration"]) & set(sp_split["evaluation"])) == 0
    
    sp_file = SPLITS_DIR / f"spanish_trackB_seed{seed}.json"
    with open(sp_file, "w") as f:
        json.dump(sp_split, f, indent=2)

print(f"Generated stratified splits across seeds: {seeds}")
print(f"OASIS-1 Primary: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")
print(f"Spanish Track B: Calibration={len(calib_df)}, Evaluation={len(eval_df)}")
print("[PASS] Cell 12: Stratified participant-level splits created without leakage.")
